# Export DenseNet121 w2.0 ke ONNX

Persis seperti model lama: opset 18, dynamic_axes=None (ukuran tetap),
do_constant_folding=True, dynamo=False. **Add Input** dulu: output notebook training.

In [1]:
import os, glob
import torch
import torch.nn as nn
from torchvision import models
device = torch.device('cpu')
print('Export di CPU (cukup untuk konversi)')

Export di CPU (cukup untuk konversi)


In [2]:
# Cari checkpoint model terbaik w2.0
kandidat = glob.glob('/kaggle/input/**/best_densenet121_w2.0.pth', recursive=True)
if kandidat:
    CHECKPOINT_PATH = kandidat[0]
else:
    semua = glob.glob('/kaggle/input/**/*.pth', recursive=True)
    CHECKPOINT_PATH = [p for p in semua if 'densenet121_w2.0' in p][0]
ONNX_PATH = '/kaggle/working/densenet121_skin.onnx'
print('Checkpoint:', CHECKPOINT_PATH)

Checkpoint: /kaggle/input/notebooks/zakiimhmmd/skinsense-training-final-1/best_densenet121_w2.0.pth


In [3]:
# Bangun arsitektur SAMA seperti training
DROPOUT = 0.4
NUM_CLASSES = 4
model = models.densenet121(weights=None)
jf = model.classifier.in_features
model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(jf, NUM_CLASSES))
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model = model.to(device)
model.eval()
print('Bobot dimuat.')

Bobot dimuat.


In [4]:
# EXPORT — RESEP PERSIS SEPERTI MODEL LAMA YANG JALAN
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,          # simpan bobot di dalam file ONNX
    opset_version=18,            # sesuai resep yang terbukti jalan
    do_constant_folding=True,    # optimasi konstanta
    input_names=['input'],
    output_names=['output'],
    dynamic_axes=None,           # KUNCI: ukuran TETAP [1,3,224,224]
    dynamo=False,                # exporter klasik (stabil untuk web)
)
print('Export selesai:', ONNX_PATH)

if os.path.exists(ONNX_PATH + '.data'):
    print('PERINGATAN: ada .onnx.data')
else:
    print('OK: single file')
print(f'Ukuran: {os.path.getsize(ONNX_PATH)/(1024*1024):.1f} MB')

/tmp/ipykernel_58/786873486.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export selesai: /kaggle/working/densenet121_skin.onnx
OK: single file
Ukuran: 26.9 MB


In [5]:
# Verifikasi
!pip install onnxruntime -q
import onnxruntime as ort
import numpy as np
sesi = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
out = sesi.run(None, {'input': np.random.randn(1,3,224,224).astype(np.float32)})
print('Model jalan. Output:', out[0].shape, '(harus (1,4))')
import onnx
m = onnx.load(ONNX_PATH)
print('Opset:', m.opset_import[0].version, '| IR:', m.ir_version)
print('\nDownload densenet121_skin.onnx dari Output.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 82.1 MB/s eta 0:00:00:00:0100:01
Model jalan. Output: (1, 4) (harus (1,4))
Opset: 18 | IR: 8

Download densenet121_skin.onnx dari Output.
